In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re 

In [3]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parents[0]
sys.path.append(str(PROJECT_ROOT))


In [4]:
print(sys.path)

['C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\python311.zip', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\DLLs', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311\\Lib', 'C:\\Users\\latha\\AppData\\Local\\Programs\\Python\\Python311', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting\\penv', '', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting\\penv\\Lib\\site-packages', 'e:\\Learning\\Data Science\\ML Project\\pds_foodgrain_forecasting']


In [5]:
# IMPORTING DEFINED FUNCTIONS

from src.text_cleaning import (
    standardise_columns,
    normalize_text_column,
    normalize_indian_state_names,
    normalize_district_name
)

from src.date_utils import (
    extract_calendar_year,
    extract_month_name,
    calculate_financial_year_from_month_raw,
    create_month_start_date,
    extract_month_number,
    extract_quarter
)

from src.geo_utils import (
    map_country_to_code,
    map_state_codes,
    expand_directional_districts,
    map_district_codes
)


In [6]:
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120) 

In [7]:
DATA_PATH = "../data/raw/"
ADDITIONAL_DATA_PATH = "../data/raw/additional_data/" 

In [8]:
df_main = pd.read_csv(DATA_PATH+"Automated & Unautomated Allocation & Distribution (Rice & Wheat).csv")

In [9]:
df_main.sample(5)

,Country,State,Year,Month,District Food Supply Office Implementing This Scheme,Commodity,"Allocated Quantity (UOM:MT(MetricTonne)), Scaling Factor:1","Allocated Quantity Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1","Allocated Quantity Without Using Electronic Point Of Sale (Epos) (UOM:MT(MetricTonne)), Scaling Factor:1"
65686,India,Madhya Pradesh,"Calendar Year (Jan - Dec), 2017","November, 2017",DSO RAISEN,Wheat,4185.70,1061.02,NaN
6197,India,Kerala,"Calendar Year (Jan - Dec), 2021","April, 2021",Thiruvananthapuram,Wheat,1792.83,1679.10,NaN
43253,India,Himachal Pradesh,"Calendar Year (Jan - Dec), 2019","February, 2019",SIRMAUR,Wheat,621.00,591.35,NaN
621,India,Uttar Pradesh,"Calendar Year (Jan - Dec), 2021","September, 2021",Ayodhya,Rice,4231.34,4081.52,NaN
46287,India,Chhattisgarh,"Calendar Year (Jan - Dec), 2018","December, 2018",RAIPUR,Rice,6928.72,6841.54,NaN


In [10]:
df_main.shape

(67522, 9)

In [11]:
df_main=standardise_columns(df_main)

In [12]:
print(df_main.columns)

Index(['country', 'state', 'year', 'month', 'district_food_supply_office_implementing_this_scheme', 'commodity',
       'allocated_quantity_uommtmetrictonne_scaling_factor1',
       'allocated_quantity_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1',
       'allocated_quantity_without_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1'],
      dtype='object')


In [13]:
# RENAMING COLUMNS
df_main = df_main.rename(columns={
    "year": "year_raw",
    "month": "month_raw",
    "district_food_supply_office_implementing_this_scheme": "dfso",
    "allocated_quantity_uommtmetrictonne_scaling_factor1": "total_allocated_qty",
    "allocated_quantity_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1": "epos_allocated_qty",
    "allocated_quantity_without_using_electronic_point_of_sale_epos_uommtmetrictonne_scaling_factor1": "not_epos_allocated_qty"
})

In [14]:
print(df_main.columns)

Index(['country', 'state', 'year_raw', 'month_raw', 'dfso', 'commodity', 'total_allocated_qty', 'epos_allocated_qty',
       'not_epos_allocated_qty'],
      dtype='object')


In [15]:
print(df_main.shape)

(67522, 9)


In [16]:
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 67522 entries, 0 to 67521
Data columns (total 9 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   country                 67522 non-null  object 
 1   state                   67522 non-null  object 
 2   year_raw                67522 non-null  object 
 3   month_raw               67522 non-null  object 
 4   dfso                    67522 non-null  object 
 5   commodity               67522 non-null  object 
 6   total_allocated_qty     67126 non-null  float64
 7   epos_allocated_qty      45145 non-null  float64
 8   not_epos_allocated_qty  22769 non-null  float64
dtypes: float64(3), object(6)
memory usage: 4.6+ MB


In [17]:
df_main.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
country,67522,1,India,67522,NaN,NaN,NaN,NaN,NaN,NaN,NaN
state,67522,35,Uttar Pradesh,11806,NaN,NaN,NaN,NaN,NaN,NaN,NaN
year_raw,67522,5,"Calendar Year (Jan - Dec), 2019",18716,NaN,NaN,NaN,NaN,NaN,NaN,NaN
month_raw,67522,58,"April, 2018",1665,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dfso,67522,1784,HAMIRPUR,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,67522,2,Rice,37605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_allocated_qty,67126.0,NaN,NaN,NaN,3999.657775,10485.358078,0.01,849.2175,2889.155,5565.0,839904.3
epos_allocated_qty,45145.0,NaN,NaN,NaN,3120.260046,3022.161033,0.01,724.41,2349.41,4643.67,49492.99
not_epos_allocated_qty,22769.0,NaN,NaN,NaN,1888.189055,9658.131083,0.01,72.2,421.4,2103.96,655202.0


In [18]:
print(df_main.duplicated().sum())

0


In [19]:
print(df_main.isna().sum())

country                       0
state                         0
year_raw                      0
month_raw                     0
dfso                          0
commodity                     0
total_allocated_qty         396
epos_allocated_qty        22377
not_epos_allocated_qty    44753
dtype: int64


In [20]:
# STANDARDISING TEXT COLUMNS
df_main["country"] = normalize_text_column(df_main["country"])
df_main["state"] = normalize_text_column(df_main["state"])
df_main["district"] = normalize_text_column(df_main["dfso"])
df_main["commodity"] = normalize_text_column(df_main["commodity"])


In [21]:
# SPECIFIC TEXT NORMALISATION 
df_main["state"] = normalize_indian_state_names(df_main["state"])
df_main["district"] = normalize_district_name(df_main["dfso"])


In [22]:
# DATE-TIME DATA EXTRACTION (FEATURE ENGINEERING)
df_main["current_year"] = extract_calendar_year(df_main["year_raw"])
df_main["month"] = extract_month_name(df_main["month_raw"])
df_main["financial_year"] = calculate_financial_year_from_month_raw(df_main["month_raw"])
df_main["date"] = create_month_start_date(df_main["month_raw"])

df_main["month_num"] = extract_month_number(df_main["date"])
df_main["quarter"] = extract_quarter(df_main["date"])


In [23]:
# SETTING COUNTRY CODE
df_main["country_code"] = map_country_to_code(df_main["country"])

In [24]:
df_main["country_code"].value_counts()


country_code
IN    67522
Name: count, dtype: int64

In [25]:
df_main[df_main["state"].isna()]["state"].unique()

array([], dtype=object)

In [26]:
df_main["state"].value_counts()

state
uttar pradesh                               11806
madhya pradesh                               7707
maharashtra                                  3948
bihar                                        3866
odisha                                       3523
gujarat                                      3262
tamil nadu                                   3128
karnataka                                    3069
west bengal                                  2320
jharkhand                                    2245
rajasthan                                    2107
telangana                                    1948
himachal pradesh                             1900
jammu and kashmir                            1812
kerala                                       1729
assam                                        1667
chhattisgarh                                 1621
arunachal pradesh                            1541
uttarakhand                                  1114
haryana                                     

In [27]:
df_main = df_main.rename(columns={"state": "state_name"})


In [28]:
# MAPPING STATE CODES
df_main = map_state_codes(df_main, state_col="state_name")


In [29]:
df_main.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'dfso', 'commodity', 'total_allocated_qty',
       'epos_allocated_qty', 'not_epos_allocated_qty', 'district', 'current_year', 'month', 'financial_year', 'date',
       'month_num', 'quarter', 'country_code', 'state_code', 'state_num_code'],
      dtype='object')

In [30]:
df_main[df_main["state_code"].isna()]["state_name"].unique()

array([], dtype=object)

In [31]:
# EXPANDING DISTRICT NAME IN CASE OF DIRECTIONAL DISTRICTS
df_main = expand_directional_districts(
    df_main,
    district_col="district",
    state_col="state_name"
)


In [32]:
# MAPPING DISTRICT CODES
df_main = map_district_codes(df_main, district_col="district")


In [33]:
df_main[df_main["full_district_code"].isna()][
    ["state_name", "dfso", "district"]
].drop_duplicates()


,state_name,dfso,district
13,arunachal pradesh,Dibang Valley,dibang valley
14,arunachal pradesh,Longding,longding
15,arunachal pradesh,Siang,siang
18,chhattisgarh,BALOD,balod
19,chhattisgarh,BALRAMPUR,balrampur
...,...,...,...
62480,karnataka,Office Name,
64556,assam,"FOOD, CIVIL SUPPLIES & CONSUMER AFFAIRS, DC OF...","dc office, kamrup, amingaon"
65008,maharashtra,"DEPUTY CONTROLLER OF RATIONING, E REGION, WADALA","deputy controller of rationing, e region, wadala"
65012,maharashtra,"DEPUTY CONTROLLER OF RATIONING, G REGION, KAND...","deputy controller of rationing, g region, kand..."


In [34]:
df_main.columns

Index(['country', 'state_name', 'year_raw', 'month_raw', 'dfso', 'commodity', 'total_allocated_qty',
       'epos_allocated_qty', 'not_epos_allocated_qty', 'district', 'current_year', 'month', 'financial_year', 'date',
       'month_num', 'quarter', 'country_code', 'state_code', 'state_num_code', 'district_code', 'full_district_code'],
      dtype='object')

In [35]:
df_main = df_main.drop(columns=[
    "country",
    "year_raw",
    "month_raw",
    "district",
    "state_num_code"
])

In [36]:
print("Missing state_code:", df_main["state_code"].isna().sum())


Missing state_code: 0


In [37]:
print("Missing district_code:", df_main["district_code"].isna().sum())

Missing district_code: 23469


In [38]:
print("Missing full_district_code:", df_main["full_district_code"].isna().sum())

Missing full_district_code: 23250


In [39]:
df_main.dtypes


state_name                        object
dfso                              object
commodity                         object
total_allocated_qty              float64
epos_allocated_qty               float64
not_epos_allocated_qty           float64
current_year                       int64
month                             object
financial_year                     int64
date                      datetime64[ns]
month_num                          int32
quarter                           object
country_code                      object
state_code                        object
district_code                     object
full_district_code                object
dtype: object

In [40]:
df_main.describe(include="all").T

,count,unique,top,freq,mean,min,25%,50%,75%,max,std
state_name,67522,34,uttar pradesh,11806,NaN,NaN,NaN,NaN,NaN,NaN,NaN
dfso,67522,1784,HAMIRPUR,135,NaN,NaN,NaN,NaN,NaN,NaN,NaN
commodity,67522,2,rice,37605,NaN,NaN,NaN,NaN,NaN,NaN,NaN
total_allocated_qty,67126.0,NaN,NaN,NaN,3999.657775,0.01,849.2175,2889.155,5565.0,839904.3,10485.358078
epos_allocated_qty,45145.0,NaN,NaN,NaN,3120.260046,0.01,724.41,2349.41,4643.67,49492.99,3022.161033
not_epos_allocated_qty,22769.0,NaN,NaN,NaN,1888.189055,0.01,72.2,421.4,2103.96,655202.0,9658.131083
current_year,67522.0,NaN,NaN,NaN,2019.206792,2017.0,2018.0,2019.0,2020.0,2021.0,1.141852
month,67522,12,april,6022,NaN,NaN,NaN,NaN,NaN,NaN,NaN
financial_year,67522.0,NaN,NaN,NaN,2018.94738,2016.0,2018.0,2019.0,2020.0,2021.0,1.158529
date,67522,NaN,NaN,NaN,2019-08-26 11:29:40.433043968,2017-01-01 00:00:00,2018-10-01 00:00:00,2019-08-01 00:00:00,2020-08-01 00:00:00,2021-10-01 00:00:00,NaN


In [41]:
print(df_main.columns)

Index(['state_name', 'dfso', 'commodity', 'total_allocated_qty', 'epos_allocated_qty', 'not_epos_allocated_qty',
       'current_year', 'month', 'financial_year', 'date', 'month_num', 'quarter', 'country_code', 'state_code',
       'district_code', 'full_district_code'],
      dtype='object')


In [42]:
# REORDERING COLUMNS
desired_column_order = [
    "country_code",
    "state_name",
    "state_code",
    "dfso",
    "district_code",
    "full_district_code",
    "current_year",
    "financial_year",
    "month",
    "month_num",
    "quarter",
    "date",
    "commodity",
    "total_allocated_qty",
    "epos_allocated_qty",
    "not_epos_allocated_qty"
]


In [43]:
existing_cols = [c for c in desired_column_order if c in df_main.columns]
df_main = df_main[existing_cols]

In [44]:
df_main.sample(5)

,country_code,state_name,state_code,dfso,district_code,full_district_code,current_year,financial_year,month,month_num,quarter,date,commodity,total_allocated_qty,epos_allocated_qty,not_epos_allocated_qty
59246,IN,himachal pradesh,HP,SHIMLA,SH,IN.HP.SH,2018,2018,april,4,2018Q2,2018-04-01,wheat,937.00,683.02,NaN
34433,IN,rajasthan,RJ,DHAULPUR,NaN,NaN,2019,2019,august,8,2019Q3,2019-08-01,wheat,4547.96,4448.23,NaN
7388,IN,jharkhand,JH,DUMKA,DU,IN.JH.DU,2021,2020,march,3,2021Q1,2021-03-01,rice,5601.72,5255.25,NaN
6545,IN,rajasthan,RJ,Ajmer Rural,NaN,NaN,2021,2021,april,4,2021Q2,2021-04-01,wheat,5543.64,5655.70,NaN
54423,IN,haryana,HR,JHAJJAR,JH,IN.HR.JH,2018,2018,july,7,2018Q3,2018-07-01,wheat,1647.69,1614.47,NaN


In [45]:
df_main.to_csv("../data/cleaned/primary_data_clean.csv", index=False)
